# 07 · Aprendizaje federado (FedAvg)

**Tema 6 del proyecto.** El notebook 04 cubrió el **aprendizaje distribuido**: un mismo dueño
reparte el cómputo entre varias GPU para entrenar más rápido. El **aprendizaje federado**
resuelve otro problema: los datos están repartidos entre varios dueños que **no pueden
compartirlos**. Ejemplo: cinco molinos arroceros, cada uno con fotos de sus granos, que quieren
un clasificador común sin entregar sus imágenes.

| | Distribuido (notebook 04) | Federado (este notebook) |
|---|---|---|
| Objetivo | Velocidad | Privacidad: los datos no salen de cada cliente |
| Datos | Centralizados, se reparten por lotes | Ya repartidos entre clientes |
| Qué viaja | Gradientes en cada paso, entre GPU de una máquina | Pesos del modelo cada ronda, por red |
| Distribución por nodo | Idéntica (IID) | A menudo distinta entre clientes (no IID) |

**Algoritmo FedAvg** (McMahan et al., 2017). En cada ronda:
1. El servidor envía el modelo global a los clientes.
2. Cada cliente entrena unas épocas **solo con sus datos**.
3. Cada cliente devuelve sus pesos (no sus imágenes).
4. El servidor promedia los pesos, ponderando por cuántas imágenes tiene cada cliente.

FedAvg se implementa aquí paso a paso para que el algoritmo quede visible. En producción se
usan frameworks como Flower o TensorFlow Federated, que hacen lo mismo sobre una red real.

**Experimentos** (5 clientes, mismo modelo y mismo total de épocas):
- **Centralizado:** todos los datos juntos, como referencia.
- **Federado IID:** cada cliente tiene una muestra representativa de las 5 variedades.
- **Federado no IID:** cada cliente tiene sobre todo unas pocas variedades (reparto con
  distribución de Dirichlet, α = 0,3), como pasaría si cada molino procesa otras variedades.

**Requisitos de la sesión:** GPU T4 x2 (se usa una) y, como Input, el dataset y
`01-preparacion-datos`. No necesita Internet.

**Salida:** `federado_rondas.csv` y `federado_resumen.csv`.

### Modo de prueba

Con `QUICK = True` todo corre en miniatura para detectar errores en pocos minutos.
**Para la corrida real ponlo en `False` antes de Save & Run All.**

In [ ]:
QUICK = True

In [ ]:
import os
import time
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SEED = 42
IMG = 64


def find_dir_with(*names, base="/kaggle/input"):
    for root, _, files in os.walk(base):
        if all(n in files for n in names):
            return Path(root)
    raise FileNotFoundError(f"No se encontró {names}: revisa los Input del notebook")


SPLITS = find_dir_with("subset_train.csv", "rice_data.py")
sub_train = pd.read_csv(SPLITS / "subset_train.csv")
sub_val = pd.read_csv(SPLITS / "subset_val.csv")
CLASSES = sub_train[["label_idx", "label"]].drop_duplicates().sort_values("label_idx")["label"].tolist()
N_CLASES = len(CLASSES)

DATA_ROOT = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if set(CLASSES) <= set(dirs):
        DATA_ROOT = Path(root)
        break

N_CLIENTES = 5
RONDAS = 2 if QUICK else 20
EPOCAS_LOCALES = 1 if QUICK else 2
if QUICK:
    sub_train = sub_train.groupby("label_idx", group_keys=False).sample(n=100, random_state=SEED)
    sub_val = sub_val.groupby("label_idx", group_keys=False).sample(n=40, random_state=SEED)
print("GPU:", tf.config.list_physical_devices("GPU"))
print(f"train={len(sub_train)}  val={len(sub_val)}  clientes={N_CLIENTES}  rondas={RONDAS}  épocas locales={EPOCAS_LOCALES}")

## 1. Datos

Se usa el subconjunto estratificado del notebook 01 (10.000 imágenes de train y 2.500 de
validación) a 64×64 píxeles, para que las muchas rondas de entrenamiento quepan en poco tiempo.

In [ ]:
def leer(path, label):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, (IMG, IMG), antialias=True), label


def cargar_arreglo(df):
    paths = (str(DATA_ROOT) + "/" + df["filepath"]).to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, df["label_idx"].to_numpy()))
    ds = ds.map(leer, num_parallel_calls=tf.data.AUTOTUNE).batch(256)
    xs, ys = zip(*[(x.numpy(), y.numpy()) for x, y in ds])
    return np.concatenate(xs).astype("float32"), np.concatenate(ys)


t0 = time.time()
X, Y = cargar_arreglo(sub_train)
X_val, Y_val = cargar_arreglo(sub_val)
print(f"Cargadas {len(Y)} + {len(Y_val)} imágenes en {time.time() - t0:.0f} s")

## 2. Reparto de los datos entre los 5 clientes

- **IID:** cada cliente recibe una quinta parte de cada variedad.
- **No IID:** para cada variedad se sortea qué fracción va a cada cliente con una distribución
  de Dirichlet (α = 0,3). Con α pequeño, cada variedad se concentra en pocos clientes.

In [ ]:
def reparto_iid(y, rng):
    clientes = [[] for _ in range(N_CLIENTES)]
    for c in range(N_CLASES):
        idx = rng.permutation(np.where(y == c)[0])
        for k, parte in enumerate(np.array_split(idx, N_CLIENTES)):
            clientes[k].extend(parte)
    return [np.array(sorted(c)) for c in clientes]


def reparto_dirichlet(y, rng, alpha=0.3):
    clientes = [[] for _ in range(N_CLIENTES)]
    for c in range(N_CLASES):
        idx = rng.permutation(np.where(y == c)[0])
        proporciones = rng.dirichlet(np.full(N_CLIENTES, alpha))
        cortes = (np.cumsum(proporciones) * len(idx)).astype(int)[:-1]
        for k, parte in enumerate(np.split(idx, cortes)):
            clientes[k].extend(parte)
    return [np.array(sorted(c)) for c in clientes]


rng = np.random.default_rng(SEED)
repartos = {"Federado IID": reparto_iid(Y, rng), "Federado no IID": reparto_dirichlet(Y, rng)}

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for ax, (nombre, rep) in zip(axes, repartos.items()):
    conteo = np.array([np.bincount(Y[idx], minlength=N_CLASES) for idx in rep])
    ax.imshow(conteo, cmap="Blues", vmin=0, vmax=conteo.max())
    for k in range(N_CLIENTES):
        for c in range(N_CLASES):
            ax.text(c, k, conteo[k, c], ha="center", va="center",
                    color="white" if conteo[k, c] > conteo.max() / 2 else "black", fontsize=9)
    ax.set_xticks(range(N_CLASES), CLASSES, rotation=20)
    ax.set_yticks(range(N_CLIENTES), [f"cliente {k + 1}" for k in range(N_CLIENTES)])
    ax.set_title(f"{nombre}: imágenes por cliente y variedad")
plt.tight_layout()
plt.show()

## 3. Modelo y FedAvg

Una CNN pequeña **sin BatchNormalization**. BatchNorm guarda estadísticas de los datos de
cada cliente, y promediarlas entre clientes con distribuciones distintas degrada el modelo
global, un problema conocido del aprendizaje federado. (El notebook 04 encontró otro problema
de BatchNorm al repartir el entrenamiento entre GPU.)

Para comparar en igualdad de condiciones, el entrenamiento centralizado recibe el mismo total
de pasadas por los datos: rondas × épocas locales.

In [ ]:
def crear_modelo():
    inp = keras.Input((IMG, IMG, 3))
    x = layers.Rescaling(1 / 255)(inp)
    for filtros in (32, 64, 128):
        x = layers.Conv2D(filtros, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filtros, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    return keras.Model(inp, layers.Dense(N_CLASES, activation="softmax")(x))


def compilar(m):
    m.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def evaluar(m):
    return m.evaluate(X_val, Y_val, batch_size=256, verbose=0)


def fedavg(reparto, nombre):
    keras.utils.set_random_seed(SEED)
    global_m = compilar(crear_modelo())
    cliente_m = crear_modelo()
    pesos_globales = global_m.get_weights()
    tamanos = np.array([len(idx) for idx in reparto], dtype=float)
    filas = []
    for r in range(1, RONDAS + 1):
        pesos_clientes = []
        for idx in reparto:
            cliente_m.set_weights(pesos_globales)       # 1. el servidor envía el modelo global
            compilar(cliente_m)                         #    (optimizador nuevo en cada ronda)
            cliente_m.fit(X[idx], Y[idx], epochs=EPOCAS_LOCALES, batch_size=32,
                          shuffle=True, verbose=0)      # 2. entrenamiento local
            pesos_clientes.append(cliente_m.get_weights())  # 3. el cliente devuelve sus pesos
        # 4. promedio ponderado por la cantidad de imágenes de cada cliente
        pesos_globales = [sum(w[i] * n for w, n in zip(pesos_clientes, tamanos)) / tamanos.sum()
                          for i in range(len(pesos_globales))]
        global_m.set_weights(pesos_globales)
        loss, acc = evaluar(global_m)
        filas.append({"experimento": nombre, "ronda": r, "val_loss": loss, "val_accuracy": acc})
        print(f"{nombre:16s} ronda {r:2d}  val_accuracy={acc:.4f}")
    return filas, global_m.count_params()


def centralizado():
    keras.utils.set_random_seed(SEED)
    m = compilar(crear_modelo())
    filas = []
    for r in range(1, RONDAS + 1):
        m.fit(X, Y, epochs=EPOCAS_LOCALES, batch_size=32, shuffle=True, verbose=0)
        loss, acc = evaluar(m)
        filas.append({"experimento": "Centralizado", "ronda": r, "val_loss": loss, "val_accuracy": acc})
        print(f"{'Centralizado':16s} ronda {r:2d}  val_accuracy={acc:.4f}")
    return filas

## 4. Entrenamiento

In [ ]:
filas, tiempos = [], {}
t0 = time.time()
filas += centralizado()
tiempos["Centralizado"] = time.time() - t0
for nombre, reparto in repartos.items():
    t0 = time.time()
    f, n_params = fedavg(reparto, nombre)
    filas += f
    tiempos[nombre] = time.time() - t0

rondas = pd.DataFrame(filas)
rondas.to_csv(OUT_DIR / "federado_rondas.csv", index=False)

## 5. Resultados

**Costo de comunicación:** en cada ronda, cada cliente descarga el modelo global y sube sus
pesos. Eso es lo que viaja por la red en lugar de las imágenes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for nombre, g in rondas.groupby("experimento", sort=False):
    axes[0].plot(g["ronda"], g["val_accuracy"], marker="o", label=nombre)
    axes[1].plot(g["ronda"], g["val_loss"], marker="o", label=nombre)
axes[0].set_ylabel("val_accuracy")
axes[1].set_ylabel("val_loss")
axes[1].set_yscale("log")
for ax in axes:
    ax.set_xlabel("Ronda (centralizado: bloques del mismo número de épocas)")
    ax.legend()
plt.suptitle("Centralizado frente a federado")
plt.tight_layout()
plt.show()

mb_modelo = n_params * 4 / 1e6
mb_datos = sum((DATA_ROOT / f).stat().st_size for f in sub_train["filepath"]) / 1e6
# La exactitud de la última ronda depende de la suerte de esa ronda; la media de las últimas 5 es más estable.
resumen = (rondas.groupby("experimento", sort=False)
           .agg(accuracy_final=("val_accuracy", "last"), mejor_accuracy=("val_accuracy", "max"),
                media_ult5=("val_accuracy", lambda s: s.tail(5).mean()),
                desv_ult5=("val_accuracy", lambda s: s.tail(5).std()),
                loss_final=("val_loss", "last")))
resumen["minutos"] = [tiempos[e] / 60 for e in resumen.index]
resumen["MB transmitidos"] = [0.0 if e == "Centralizado" else mb_modelo * 2 * N_CLIENTES * RONDAS
                              for e in resumen.index]
resumen["imágenes compartidas"] = [len(Y) if e == "Centralizado" else 0 for e in resumen.index]
resumen["MB de imágenes compartidas"] = [mb_datos if e == "Centralizado" else 0.0 for e in resumen.index]
resumen.to_csv(OUT_DIR / "federado_resumen.csv")
print(f"Tamaño del modelo: {mb_modelo:.2f} MB ({n_params:,} parámetros)")
print(f"Tamaño de las {len(Y)} imágenes originales (JPEG): {mb_datos:.1f} MB")
resumen.round(4)

**Cómo leer los resultados:**

- **Centralizado frente a federado IID:** si quedan parecidos, FedAvg logra casi el mismo
  modelo sin que ningún cliente comparta sus imágenes. Es el resultado esperado cuando todos los
  clientes tienen datos parecidos.
- **Federado no IID:** suele quedar por debajo y oscilar más entre rondas. Cada cliente empuja
  el modelo hacia sus propias variedades durante el entrenamiento local (*client drift*), y el
  promedio no siempre es un buen compromiso. Es el principal desafío del aprendizaje federado
  real, y por eso existen variantes como FedProx o SCAFFOLD.
- **Estabilidad:** la exactitud de la última ronda puede caer por azar, sobre todo en el caso
  no IID. Por eso la tabla incluye la media y la desviación de las últimas 5 rondas.
- **Costo:** ninguna imagen sale de su cliente; esa es la ventaja de privacidad. Pero el tráfico
  no es gratis: cada envío del modelo pesa menos que los datos, y aun así 20 rondas × 5 clientes
  × 2 direcciones pueden sumar más MB que enviar las imágenes una sola vez (comparar las dos
  últimas columnas de la tabla). El federado paga en comunicación lo que gana en privacidad. Con
  muchos más datos por cliente (millones de imágenes) el balance se invierte a su favor.